# Kiểm chứng lựa chọn Loss Function bằng Validation (chống rò rỉ)

**Dự án Tốt nghiệp - Energy Forecasting - Nhóm The Outliers**

## 1. Mục đích

Notebook này **KHÔNG train lại gì cả** — chỉ load lại 6 model `.pkl` đã train sẵn (MAE/Huber/MSE × H1/H4) và tính WAPE thật trên tập **validation** (chưa từng đụng test), để xem model nào thắng thật.

Lý do cần làm: phát hiện lỗi rò rỉ dữ liệu — bước chọn loss function trước đây đang dùng nhầm tập test thay vì validation (`metrics_val.json` bị ghi từ metric tính trên `test_h`). Notebook này verify lại bằng validation thật, không đụng test, chạy trong vài giây.

## 2. Import và cấu hình đường dẫn

In [2]:
import os
import json
import pickle
import numpy as np
import pandas as pd

BASE = '/home/tandat/Desktop/Du_An_Tot_Nghiep_v3'
FOLDS_DIR = f'{BASE}/data/model/v3/05_selected/time_series_folds'
TRAIN_DIR = f'{BASE}/data/model/v3/06_train'
EPS_ELEV = 0.05
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'

print('Da cau hinh duong dan. FOLDS_DIR =', FOLDS_DIR)
print('TRAIN_DIR =', TRAIN_DIR)

Da cau hinh duong dan. FOLDS_DIR = /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v3/05_selected/time_series_folds
TRAIN_DIR = /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v3/06_train


## 3. Danh sách cột dịch theo horizon (`_mt`)

Sao chép đúng từ `04_1_train_mae.py` (COT_TAT_DINH) — các cột này được dịch tới thời điểm T+h thành cột mới hậu tố `_mt`, giống hệt hàm `them_muc_tieu()` trong pipeline training thật.

In [3]:
COT_TAT_DINH = [
    'solar_elevation', 'solar_azimuth', 'azimuth_sin', 'azimuth_cos', 'sin_elevation',
    'ghi_cs', 'clearsky_proxy', 'ky_vong', 'ty_le_bao_hoa',
    'minute_of_day', 'hour_of_day', 'hour_bucket_model', 'hour', 'hour_sin', 'hour_cos',
    'minute', 'day', 'day_of_week', 'month', 'day_of_year', 'doy_sin', 'doy_cos',
]
print(f'Co {len(COT_TAT_DINH)} cot se duoc dich thanh dac trung _mt.')

Co 22 cot se duoc dich thanh dac trung _mt.


## 4. Hàm chuẩn hoá mục tiêu và tính WAPE

Giống hệt `mau_chuan_hoa()` trong script training gốc.

In [4]:
def mau_chuan_hoa(df):
    """Mau so de chuan hoa muc tieu: quy mo tram nhan sin(goc cao mat troi)."""
    return (df['site_scale'] * np.clip(df['sin_elevation'], EPS_ELEV, None)).to_numpy()


def compute_wape(yt, yp):
    yt = np.asarray(yt, dtype=float)
    yp = np.asarray(yp, dtype=float)
    denom = np.sum(np.abs(yt))
    return float(np.sum(np.abs(yt - yp)) / denom * 100.0) if denom > 0 else float('nan')

print('Da dinh nghia mau_chuan_hoa va compute_wape.')

Da dinh nghia mau_chuan_hoa va compute_wape.


## 5. Nạp lại các fold validation

Đọc trực tiếp `fold_{n}_val_selected.parquet` (tập validation thật, KHÔNG phải test), tự tạo `y_true` (dịch target theo horizon, giống `them_muc_tieu()`) và các cột `_mt` cần thiết, không dùng lại bất kỳ dòng nào của tập test.

In [5]:
def load_val_folds(features, horizon_steps):
    frames = []
    n = 1
    while True:
        path = f'{FOLDS_DIR}/fold_{n}_val_selected.parquet'
        if not os.path.exists(path):
            break
        # can doc them cac cot GOC (khong _mt) cua COT_TAT_DINH de dich thanh _mt
        base_needed = [c[:-3] if c.endswith('_mt') and c[:-3] in COT_TAT_DINH else c for c in features]
        need = list(dict.fromkeys(
            base_needed + [SITE_COL, TIMESTAMP_COL, TARGET_COL, 'site_scale', 'sin_elevation',
                           'tran_cong_suat', 'energy_source', 'is_daylight']))
        d = pd.read_parquet(path)
        need = [c for c in need if c in d.columns]
        d = d[need].sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)
        h = int(horizon_steps)
        d['y_true'] = d.groupby(SITE_COL)[TARGET_COL].shift(-h)
        g = d.groupby(SITE_COL)
        for c in COT_TAT_DINH:
            if c in d.columns and f'{c}_mt' in features:
                d[f'{c}_mt'] = g[c].shift(-h)
        d = d.dropna(subset=['y_true'])
        d = d[(d['site_scale'] > 0) & (d['sin_elevation'] > EPS_ELEV)].copy()
        frames.append(d)
        n += 1
    if not frames:
        raise FileNotFoundError(f'Khong tim thay fold validation nao trong {FOLDS_DIR}')
    return pd.concat(frames, ignore_index=True)

print('Da dinh nghia load_val_folds.')

Da dinh nghia load_val_folds.


## 6. Đánh giá 1 model (load lại `.pkl`, dự báo trên validation, tính metric)

In [6]:
_VAL_CACHE = {}


def _metric_1_scope(yt, yp):
    yt = np.asarray(yt, dtype=float); yp = np.asarray(yp, dtype=float)
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2))) if len(yt) else float('nan')
    mae = float(np.mean(np.abs(yt - yp))) if len(yt) else float('nan')
    ss_res = np.sum((yt - yp) ** 2); ss_tot = np.sum((yt - np.mean(yt)) ** 2)
    r2 = float(1 - ss_res / ss_tot) if len(yt) and ss_tot > 0 else float('nan')
    return {'wape': compute_wape(yt, yp), 'rmse': rmse, 'mae': mae, 'r2': r2, 'n': int(len(yt))}


def eval_one(loss_name, h_label):
    cfg_path = f'{TRAIN_DIR}/{loss_name}/{h_label}/model_config.json'
    pkl_path = f'{TRAIN_DIR}/{loss_name}/{h_label}/model.pkl'
    cfg = json.load(open(cfg_path))
    features = cfg['features']
    medians = cfg['feature_medians']
    horizon_steps = cfg['horizon_steps']

    with open(pkl_path, 'rb') as f:
        model = pickle.load(f)

    cache_key = int(horizon_steps)
    if cache_key not in _VAL_CACHE:
        _VAL_CACHE[cache_key] = load_val_folds(features, horizon_steps)
    val = _VAL_CACHE[cache_key]

    missing_feats = [c for c in features if c not in val.columns]
    if missing_feats:
        raise KeyError(f'Thieu {len(missing_feats)} dac trung: {missing_feats[:5]}...')

    X = val[features].fillna(pd.Series(medians)).astype(np.float32)
    k_pred = np.clip(model.predict(X), 0, 1.5)
    y_pred = np.minimum(k_pred * mau_chuan_hoa(val), val['tran_cong_suat'].to_numpy() * 1.02)
    y_pred = np.where(val['sin_elevation'].to_numpy() <= EPS_ELEV, 0.0, y_pred)
    y_true = val['y_true'].to_numpy()

    scope_all = _metric_1_scope(y_true, y_pred)

    mask = np.ones(len(val), dtype=bool)
    if 'energy_source' in val.columns:
        mask &= (val['energy_source'] == 'measured').to_numpy()
    if 'is_daylight' in val.columns:
        mask &= val['is_daylight'].fillna(False).astype(bool).to_numpy()
    scope_md = _metric_1_scope(y_true[mask], y_pred[mask])

    # Ghi de truc tiep metrics_val.json THAT (thay cho file cu tinh sai tren tap test),
    # de notebook 07 doc dung ngay lan chay tiep theo, khong can chay lai 06_1/2/3.
    out_dir = f'{TRAIN_DIR}/{loss_name}/{h_label}'
    with open(f'{out_dir}/metrics_val.json', 'w', encoding='utf-8') as f:
        json.dump({'horizon_steps': int(horizon_steps), 'loss_name': loss_name,
                   'feature_set_name': '', 'measured_daylight': scope_md,
                   'all': scope_all}, f, indent=2, ensure_ascii=False, default=str)

    return {'loss': loss_name, 'horizon': h_label, 'n_rows_val_total': len(val),
            'n_measured_daylight': scope_md['n'], 'wape_val_%': round(scope_md['wape'], 4),
            'rmse_val': round(scope_md['rmse'], 4), 'mae_val': round(scope_md['mae'], 4),
            'r2_val': round(scope_md['r2'], 4)}

print('Da dinh nghia eval_one (ban moi - tu ghi de metrics_val.json that).')

Da dinh nghia eval_one (ban moi - tu ghi de metrics_val.json that).


**Lưu ý:** `eval_one()` ở trên giờ **tự động ghi đè** `metrics_val.json` thật vào đúng thư mục `data/model/v3/06_train/<loss>/<h>/` khi được gọi ở Bước 7 bên dưới — không cần chạy lại `06_1/06_2/06_3` để cập nhật file này.

## 7. Chạy đánh giá cho cả 6 model và xác định model thắng thật

In [7]:
rows = []
for loss in ['mae', 'huber', 'mse']:
    for h in ['h1', 'h4']:
        try:
            r = eval_one(loss, h)
            rows.append(r)
            print(f"{loss:6s} {h}: WAPE_val={r['wape_val_%']:>8.4f}%  RMSE_val={r['rmse_val']:>8.4f}  "
                  f"MAE_val={r['mae_val']:>8.4f}  R2_val={r['r2_val']:>7.4f}  "
                  f"(n={r['n_measured_daylight']:,} / {r['n_rows_val_total']:,})")
        except Exception as e:
            print(f'{loss:6s} {h}: LOI - {e}')

df_ket_qua = pd.DataFrame(rows)
display(df_ket_qua)

mae    h1: WAPE_val= 17.0485%  RMSE_val=  2.8373  MAE_val=  1.1259  R2_val= 0.9312  (n=866,826 / 987,330)
mae    h4: WAPE_val= 20.9069%  RMSE_val=  3.2785  MAE_val=  1.3619  R2_val= 0.9090  (n=866,589 / 987,087)
huber  h1: WAPE_val= 14.6745%  RMSE_val=  2.2520  MAE_val=  0.9691  R2_val= 0.9567  (n=866,826 / 987,330)
huber  h4: WAPE_val= 17.1442%  RMSE_val=  2.4904  MAE_val=  1.1168  R2_val= 0.9475  (n=866,589 / 987,087)
mse    h1: WAPE_val= 15.2906%  RMSE_val=  2.3415  MAE_val=  1.0098  R2_val= 0.9531  (n=866,826 / 987,330)
mse    h4: WAPE_val= 17.2775%  RMSE_val=  2.5246  MAE_val=  1.1254  R2_val= 0.9461  (n=866,589 / 987,087)


,loss,horizon,n_rows_val_total,n_measured_daylight,wape_val_%,rmse_val,mae_val,r2_val
0,mae,h1,987330,866826,17.0485,2.8373,1.1259,0.9312
1,mae,h4,987087,866589,20.9069,3.2785,1.3619,0.9090
2,huber,h1,987330,866826,14.6745,2.2520,0.9691,0.9567
3,huber,h4,987087,866589,17.1442,2.4904,1.1168,0.9475
4,mse,h1,987330,866826,15.2906,2.3415,1.0098,0.9531
5,mse,h4,987087,866589,17.2775,2.5246,1.1254,0.9461


In [8]:
print('=== MODEL THANG THAT TREN VALIDATION (WAPE thap nhat, khong dung tap test) ===')
nguoi_thang = {}
for h in ['h1', 'h4']:
    sub = [r for r in rows if r['horizon'] == h]
    if sub:
        best = min(sub, key=lambda r: r['wape_val_%'])
        nguoi_thang[h] = best['loss']
        print(f"{h}: {best['loss'].upper()} thang voi WAPE_val = {best['wape_val_%']:.4f}%")
print()
print('Ket qua nay dung de CHON model, khong dung de bao cao headline.')
print('So lieu headline chinh thuc van lay tu ket_qua.json / metrics_overall.json tren tap TEST')
print('cua dung model vua thang o day (chi doc 1 lan, khong dung de chon lai).')

=== MODEL THANG THAT TREN VALIDATION (WAPE thap nhat, khong dung tap test) ===
h1: HUBER thang voi WAPE_val = 14.6745%
h4: HUBER thang voi WAPE_val = 17.1442%

Ket qua nay dung de CHON model, khong dung de bao cao headline.
So lieu headline chinh thuc van lay tu ket_qua.json / metrics_overall.json tren tap TEST
cua dung model vua thang o day (chi doc 1 lan, khong dung de chon lai).


## 8. Xuất kết quả ra file (để trích dẫn trong report)

In [9]:
OUT_PATH = f'{BASE}/data/model/v3/07_final_test/val_model_selection_check.json'
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump({'ket_qua_tung_model': rows, 'model_thang': nguoi_thang}, f, indent=2, ensure_ascii=False)
print(f'Da luu: {OUT_PATH}')

Da luu: /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v3/07_final_test/val_model_selection_check.json
